In [1]:
!pip install pyspark==3.5.0 delta-spark==3.1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.2 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=31b116fb0ab8d46689806163ab194e17f8d7c039665bb74146e0bd601014b3b3
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("MedallionLakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark

In [6]:
import requests

url = "https://raw.githubusercontent.com/plotly/datasets/master/uber-rides-data1.csv"
file_path = "/content/uber.csv"

response = requests.get(url)

with open(file_path, "wb") as f:
    f.write(response.content)

print("Download complete")

Download complete


In [7]:
df_raw = spark.read.csv(file_path, header=True, inferSchema=True)

df_raw.show(5)

+-------------------+-------+--------+
|          Date/Time|    Lat|     Lon|
+-------------------+-------+--------+
|2014-04-01 00:11:00| 40.769|-73.9549|
|2014-04-01 00:17:00|40.7267|-74.0345|
|2014-04-01 00:21:00|40.7316|-73.9873|
|2014-04-01 00:28:00|40.7588|-73.9776|
|2014-04-01 00:33:00|40.7594|-73.9722|
+-------------------+-------+--------+
only showing top 5 rows



In [8]:
bronze_path = "/content/bronze"

df_raw.write.format("delta").mode("overwrite").save(bronze_path)

In [10]:
from pyspark.sql.functions import col

df_bronze = spark.read.format("delta").load(bronze_path)

df_silver = df_bronze \
    .dropna() \
    .filter(col("Lat").isNotNull()) \
    .filter(col("Lon").isNotNull())

silver_path = "/content/silver"

df_silver.write.format("delta").mode("overwrite").save(silver_path)

df_silver.show(5)

+-------------------+-------+--------+
|          Date/Time|    Lat|     Lon|
+-------------------+-------+--------+
|2014-04-01 00:11:00| 40.769|-73.9549|
|2014-04-01 00:17:00|40.7267|-74.0345|
|2014-04-01 00:21:00|40.7316|-73.9873|
|2014-04-01 00:28:00|40.7588|-73.9776|
|2014-04-01 00:33:00|40.7594|-73.9722|
+-------------------+-------+--------+
only showing top 5 rows



In [11]:
from pyspark.sql.functions import count, to_date, col

df_silver = spark.read.format("delta").load(silver_path)

df_gold = df_silver \
    .withColumn("date", to_date(col("Date/Time"))) \
    .groupBy("date") \
    .agg(
        count("*").alias("total_trips")
    )

gold_path = "/content/gold"

df_gold.write.format("delta").mode("overwrite").save(gold_path)

df_gold.show()

+----------+-----------+
|      date|total_trips|
+----------+-----------+
|2014-04-29|      22835|
|2014-04-15|      20641|
|2014-04-18|      18074|
|2014-05-11|      14901|
|2014-05-01|      23375|
|2014-05-14|      22218|
|2014-04-08|      16188|
|2014-05-07|      21891|
|2014-05-24|      14651|
|2014-04-05|      19521|
|2014-04-25|      25095|
|2014-04-16|      17717|
|2014-05-28|      22240|
|2014-04-27|      14677|
|2014-05-02|      24235|
|2014-05-29|      24930|
|2014-05-30|      24413|
|2014-05-31|      21261|
|2014-04-11|      20420|
|2014-04-20|      11017|
+----------+-----------+
only showing top 20 rows



In [12]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(gold_path)

df_old.show(5)

+----------+-----------+
|      date|total_trips|
+----------+-----------+
|2014-04-29|      22835|
|2014-04-15|      20641|
|2014-04-18|      18074|
|2014-05-11|      14901|
|2014-05-01|      23375|
+----------+-----------+
only showing top 5 rows



In [15]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# Load source
new_data = spark.read.format("delta").load(silver_path)

# remove duplicates BEFORE merge
new_data_clean = new_data.dropDuplicates(["Date/Time", "Lat", "Lon"])

# Load Delta table
delta_table = DeltaTable.forPath(spark, silver_path)

# Merge safely
delta_table.alias("old") \
    .merge(
        new_data_clean.alias("new"),
        "old.`Date/Time` = new.`Date/Time` AND old.Lat = new.Lat AND old.Lon = new.Lon"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

In [16]:
from pyspark.sql.functions import sha2, concat_ws

df = new_data.withColumn(
    "record_id",
    sha2(concat_ws("-", "Date/Time", "Lat", "Lon"), 256)
)

In [17]:
from pyspark.sql.functions import hour, col

df = spark.read.format("delta").load(silver_path)

df_evolved = df \
    .withColumn("hour", hour(col("Date/Time"))) \
    .withColumn("is_peak_hour", (hour(col("Date/Time")) >= 7) & (hour(col("Date/Time")) <= 10))

df_evolved.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(silver_path)

df_evolved.show(5)

+-------------------+-------+--------+----+------------+
|          Date/Time|    Lat|     Lon|hour|is_peak_hour|
+-------------------+-------+--------+----+------------+
|2014-04-01 00:00:00|40.7188|-73.9863|   0|       false|
|2014-04-01 00:01:00|40.7355|-73.9966|   0|       false|
|2014-04-01 00:02:00|40.7556|-73.9874|   0|       false|
|2014-04-01 00:05:00|40.7572|-73.9781|   0|       false|
|2014-04-01 00:06:00|40.7308|-73.9808|   0|       false|
+-------------------+-------+--------+----+------------+
only showing top 5 rows

